# 04 · Hybrid Fusion — Combining the Text and Network Signals

**AEGIS-SN** — the layer that makes the two branches one detector.

## Why fuse at all

The two branches fail in opposite directions, which is the whole argument:

| | text branch (02) | graph branch (03) |
|---|---|---|
| **sees** | what an account says | how a population behaves |
| **strong on** | payloads, injections, clumsy generation | rings, synchrony, copy-paste swarms |
| **blind to** | a fluent agent posting ordinary sentences | a lone account with no interaction history |
| **degrades when** | the attacker swaps generator (02 §8) | there is no population to compare against |

Neither is sufficient. A 2026 agent that writes well defeats the text branch; a single
sleeper account defeats the graph branch. An attacker has to beat **both** simultaneously,
and the fusion layer is what forces that.

## Late fusion, not joint training

The two branches are trained separately and combined at the score level. Three reasons,
and the third is the one that decides it:

1. **Auditability.** An analyst can be shown *"flagged because the text scored 0.31 but the
   account is in a 9-member synchrony cluster"*. A jointly-trained end-to-end model gives
   one number and no account of itself.
2. **Different latencies.** Text is available the instant a post arrives. Network evidence
   needs an observation window. In deployment they genuinely do not arrive together.
3. **Different units of analysis.** The text branch scores a *post*; the graph branch
   scores an *account*. §3 is where that mismatch gets resolved, and it is the real work
   of this notebook.

## Prerequisites

Notebooks 01–03. This notebook reads only `data/processed/`.

## 1 · Environment

In [1]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd()
for _candidate in (_here, *_here.parents):
    if (_candidate / "ml" / "src" / "aegis").is_dir():
        sys.path.insert(0, str(_candidate / "ml" / "src"))
        break
else:
    raise RuntimeError("Could not locate ml/src/aegis — launch Jupyter from the repo root.")

from aegis import config as acfg
from aegis import io_utils as iou
from aegis import metrics as amx
from aegis import viz

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)

settings = acfg.load_config()
acfg.set_seed(settings.seed)

FCFG = settings.fusion_model
FBETA = float(FCFG.get("fbeta", 1.5))
BANDS = FCFG.get("decision_thresholds", {"monitor": 0.35, "investigate": 0.60, "escalate": 0.82})
WEIGHTS = FCFG.get("weighted_baseline", {"w_text": 0.45, "w_graph": 0.55})

print(f"strategy     : {FCFG.get('strategy', 'stacking')}")
print(f"meta-learner : {FCFG.get('meta_learner', 'logistic')}")
print(f"calibration  : {FCFG.get('calibrate', 'isotonic')}")
print(f"F-beta       : {FBETA}")
print(f"triage bands : {BANDS}")

11:46:51 │ INFO    │ aegis │ AEGIS-SN config loaded from C:\Users\dabhi\Documents\Major-Project\Complete-project\ml\configs\default.yaml


11:46:54 │ INFO    │ aegis │ root=C:\Users\dabhi\Documents\Major-Project\Complete-project | seed=42 | smoke_test=True | device=cpu


11:46:54 │ WARNING │ aegis │ SMOKE_TEST is ON: datasets capped at 1500 rows and epochs reduced. Set AEGIS_SMOKE_TEST=0 for a publication run.


strategy     : stacking
meta-learner : logistic
calibration  : isotonic
F-beta       : 1.5
triage bands : {'monitor': 0.35, 'investigate': 0.6, 'escalate': 0.82}


## 2 · Load both branches' outputs

In [2]:
text_val = iou.load_frame(settings.paths.processed / "text_scores_validation.parquet")
text_test = iou.load_frame(settings.paths.processed / "text_scores_test.parquet")
graph_scores = iou.load_frame(settings.paths.processed / "graph_scores.parquet")
campaign_graph = iou.load_frame(settings.paths.processed / "campaign_graph_scores.parquet")
campaign_posts = iou.load_frame(settings.paths.processed / "campaign_posts.parquet")

text_meta = json.loads((settings.paths.text_model / "text_metrics.json").read_text())
graph_meta = json.loads((settings.paths.graph_model / "graph_metrics.json").read_text())

print(f"text  val/test : {len(text_val):,} / {len(text_test):,} posts")
print(f"graph accounts : {len(graph_scores):,}")
print(f"campaign       : {len(campaign_graph):,} accounts, {len(campaign_posts):,} posts")
print(f"\ntext  threshold: {text_meta['threshold']:.3f}   test F1 {text_meta['test']['f1']:.4f}")
if graph_meta.get("gnn_test"):
    print(f"graph threshold: {graph_meta['threshold']:.3f}   test F1 {graph_meta['gnn_test']['f1']:.4f}")

text  val/test : 738 / 727 posts
graph accounts : 2,074
campaign       : 48 accounts, 327 posts

text  threshold: 0.502   test F1 0.6882
graph threshold: 0.500   test F1 0.9651


## 3 · The unit-of-analysis problem

The text branch scores **posts**. The graph branch scores **accounts**. Fusing them means
picking a common unit, and the choice is not neutral — it is the most consequential
modelling decision in this notebook, so it is worth being explicit about.

**The unit is the account**, because that is what an analyst acts on. Nobody suspends a
sentence.

So post-level text scores have to be aggregated per account. The aggregator matters:

* `mean` — dilutes. An agent posting 199 innocuous updates and one payload gets a low mean,
  and that is exactly the evasion strategy a competent operator would use.
* `max` — the opposite failure. Over 200 posts the maximum of *any* score distribution
  drifts toward 1.0, so prolific accounts look guilty regardless of content.
* **`p90` and `top-k mean`** — a compromise: sensitive to a small number of bad posts,
  but not to a single outlier. This is what we use.

All of them are computed, and §4 lets the meta-learner decide how to weigh them rather
than committing to one a priori.

In [3]:
def aggregate_text_to_account(posts: pd.DataFrame, *, score_col: str, user_col: str) -> pd.DataFrame:
    """Collapse post-level text scores into one row per account."""
    grouped = posts.groupby(user_col)[score_col]
    out = pd.DataFrame({
        "text_mean": grouped.mean(),
        "text_max": grouped.max(),
        "text_p90": grouped.quantile(0.90),
        "text_std": grouped.std().fillna(0.0),
        "n_posts": grouped.size(),
    })
    # Mean of the worst 3 posts: robust to one fluke, sensitive to a small cluster
    # of payloads, which is the shape an evasive agent's output actually has.
    out["text_top3_mean"] = grouped.apply(lambda s: s.nlargest(min(3, len(s))).mean())
    # Share of posts over the text branch's own tuned threshold — "how much of this
    # account's output is adversarial", which is more interpretable than any moment.
    out["text_flag_rate"] = grouped.apply(lambda s: float((s >= text_meta["threshold"]).mean()))
    return out.reset_index().rename(columns={user_col: "user_id"})

### The evaluation set for fusion

There is a genuine gap here and it should be stated rather than papered over: the text
corpus (HC3, M4, WildJailbreak, …) and the graph corpus (Cresci-2017) **describe different
accounts**. No public dataset gives us both branches' evidence over one population of real
accounts — that is precisely the gap the synthetic campaign exists to fill.

So fusion is evaluated in two places, and they answer different questions:

* **§4–6 · the synthetic 2026 campaign** — the only set where both branches see the *same*
  accounts. This is where the fusion model is fitted and where its headline number comes
  from. It is generated data, and is labelled as such.
* **§7 · Cresci accounts with a text proxy** — real accounts, with the text branch scoring
  their actual tweets. This checks the fusion holds up on real data.

In [4]:
campaign_text = campaign_posts.loc[:, ["user_id", "text", "is_agent", "carries_injection"]].copy()

_text_scores_path = settings.paths.processed / "campaign_text_scores.parquet"
if _text_scores_path.exists():
    campaign_text = iou.load_frame(_text_scores_path)
    print(f"loaded cached campaign text scores: {len(campaign_text):,}")
else:
    # Score the campaign posts with the fine-tuned model from notebook 02. Falls back
    # to the lexical injection heuristic if torch/transformers are unavailable, so the
    # fusion logic below stays runnable without a GPU.
    try:
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        _tok = AutoTokenizer.from_pretrained(str(settings.paths.text_model))
        _mdl = AutoModelForSequenceClassification.from_pretrained(str(settings.paths.text_model))
        _mdl.eval()
        _scores = []
        with torch.no_grad():
            for _i in range(0, len(campaign_text), 32):
                _batch = campaign_text["text"].iloc[_i:_i + 32].astype(str).tolist()
                _enc = _tok(_batch, truncation=True, max_length=int(settings.text_model.get("max_length", 256)),
                            padding=True, return_tensors="pt")
                _scores.append(torch.softmax(_mdl(**_enc).logits, dim=-1)[:, 1].numpy())
        campaign_text["text_score"] = np.concatenate(_scores)
        print(f"scored {len(campaign_text):,} campaign posts with the fine-tuned model")
    except Exception as exc:
        from aegis.text_utils import injection_lexical_score

        print(f"transformer scoring unavailable ({type(exc).__name__}) — using the lexical proxy.")
        print("Numbers below are then a LOWER BOUND on the text branch's contribution.")
        campaign_text["text_score"] = campaign_text["text"].map(injection_lexical_score)
    iou.save_frame(campaign_text, _text_scores_path)

campaign_text_agg = aggregate_text_to_account(campaign_text, score_col="text_score", user_col="user_id")
print(f"\naggregated to {len(campaign_text_agg):,} accounts")
print(campaign_text_agg.describe().round(3).to_string())

loaded cached campaign text scores: 327

aggregated to 48 accounts


       text_mean  text_max  text_p90  text_std  n_posts  text_top3_mean  text_flag_rate
count     48.000    48.000    48.000      48.0   48.000          48.000          48.000
mean       0.502     0.502     0.502       0.0    6.812           0.502           0.824
std        0.000     0.000     0.000       0.0    4.680           0.000           0.229
min        0.502     0.502     0.502       0.0    2.000           0.502           0.000
25%        0.502     0.502     0.502       0.0    3.000           0.502           0.750
50%        0.502     0.502     0.502       0.0    6.000           0.502           0.888
75%        0.502     0.502     0.502       0.0    9.000           0.502           1.000
max        0.502     0.502     0.502       0.0   23.000           0.502           1.000


In [5]:
fusion = campaign_graph.merge(campaign_text_agg, on="user_id", how="left")

# Accounts with no posts in the window have no text evidence. Filling with 0.0 would
# assert "definitely benign", which is a claim we cannot make; the neutral 0.5 says
# "no information", and `has_text` lets the meta-learner learn what absence means.
_text_cols = ["text_mean", "text_max", "text_p90", "text_top3_mean", "text_flag_rate", "text_std"]
fusion["has_text"] = fusion["text_mean"].notna().astype(int)
for _c in _text_cols:
    fusion[_c] = fusion[_c].fillna(0.5 if _c != "text_std" else 0.0)
fusion["n_posts"] = fusion["n_posts"].fillna(0)

print(f"fusion frame: {fusion.shape}")
print(f"accounts with text evidence: {int(fusion['has_text'].sum())} of {len(fusion)}")
print(f"label balance: {fusion['label'].value_counts().to_dict()}")

fusion frame: (48, 28)
accounts with text evidence: 48 of 48
label balance: {0: 40, 1: 8}


### Do the two signals actually disagree?

If text and graph scores were perfectly correlated, fusion would be pointless — you would
just pick the better one. **Low correlation is the good outcome here**: it means the
branches are seeing different things, and combining them can beat either.

In [6]:
_corr = fusion[["graph_score", "text_p90", "text_max", "text_mean"]].corr(method="spearman")
print("Spearman correlation between branch scores:")
print(_corr.round(3).to_string())
print(f"\ngraph vs text_p90: {_corr.loc['graph_score', 'text_p90']:.3f}")
print(
    "Near zero means the branches are near-independent, so the fusion has genuine"
    "\nheadroom. Near 1.0 would mean one of them is redundant."
)

print("\nmean scores by true class:")
print(
    fusion.groupby("label")[["graph_score", "text_p90", "text_max", "text_flag_rate"]]
    .mean().round(3).to_string()
)

Spearman correlation between branch scores:
             graph_score  text_p90  text_max  text_mean
graph_score        1.000     0.256     0.192      0.262
text_p90           0.256     1.000     0.929      0.788
text_max           0.192     0.929     1.000      0.658
text_mean          0.262     0.788     0.658      1.000

graph vs text_p90: 0.256
Near zero means the branches are near-independent, so the fusion has genuine
headroom. Near 1.0 would mean one of them is redundant.

mean scores by true class:
       graph_score  text_p90  text_max  text_flag_rate
label                                                 
0              0.0     0.502     0.502           0.796
1              0.0     0.502     0.502           0.964


### Where each branch is alone

The four quadrants: which accounts does each branch catch that the other misses? The
off-diagonal cells are the entire justification for the hybrid.

In [7]:
_gt = graph_meta.get("threshold") or 0.5
_tt = text_meta["threshold"]
fusion["graph_flag"] = (fusion["graph_score"] >= _gt).astype(int)
fusion["text_flag"] = (fusion["text_p90"] >= _tt).astype(int)

print("detection quadrants (true agents only):")
_agents = fusion[fusion["label"] == 1]
print(pd.crosstab(_agents["text_flag"], _agents["graph_flag"],
                  rownames=["text flags"], colnames=["graph flags"]).to_string())
print(f"\ncaught by text only  : {int(((_agents['text_flag'] == 1) & (_agents['graph_flag'] == 0)).sum())}")
print(f"caught by graph only : {int(((_agents['text_flag'] == 0) & (_agents['graph_flag'] == 1)).sum())}")
print(f"caught by both       : {int(((_agents['text_flag'] == 1) & (_agents['graph_flag'] == 1)).sum())}")
print(f"missed by both       : {int(((_agents['text_flag'] == 0) & (_agents['graph_flag'] == 0)).sum())}")

print("\nfalse positives on organic accounts:")
_organic = fusion[fusion["label"] == 0]
print(pd.crosstab(_organic["text_flag"], _organic["graph_flag"],
                  rownames=["text flags"], colnames=["graph flags"]).to_string())

detection quadrants (true agents only):
graph flags  0
text flags    
1            8

caught by text only  : 8
caught by graph only : 0
caught by both       : 0
missed by both       : 0

false positives on organic accounts:
graph flags   0
text flags     
0             2
1            38


## 4 · Three fusion strategies

Compared head to head, simplest first. Complexity has to earn its place.

1. **Weighted average** — `0.45·text + 0.55·graph` from the config. No fitting at all.
   Graph is weighted higher because 03 §8 showed it transfers better across eras.
2. **Rule-based OR** — flag if either branch fires. Maximises recall, and its precision
   is the argument against it.
3. **Stacking** — a logistic meta-learner over both branches' scores plus the aggregation
   variants. This is the configured default.

With a small campaign the meta-learner is fitted with **cross-validated predictions** and
reported via stratified CV rather than a single split, because a single held-out fold of a
48-account graph is a handful of rows and any number from it is noise.

In [8]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

y = fusion["label"].to_numpy()

strategies: dict[str, np.ndarray] = {}

# --- 1. weighted average ---------------------------------------------------
strategies["weighted_avg"] = (
    float(WEIGHTS["w_text"]) * fusion["text_p90"].to_numpy()
    + float(WEIGHTS["w_graph"]) * fusion["graph_score"].to_numpy()
)

# --- 2. rule-based OR ------------------------------------------------------
strategies["rule_or"] = np.maximum(
    fusion["text_p90"].to_numpy(), fusion["graph_score"].to_numpy()
)

# --- 3. stacking -----------------------------------------------------------
META_FEATURES = [
    "graph_score", "rf_score",
    "text_p90", "text_max", "text_mean", "text_top3_mean", "text_flag_rate", "text_std",
    "has_text", "n_posts",
    # A few raw coordination features go in directly: the meta-learner can then
    # express "high text score AND high synchrony" rather than only a linear blend
    # of two summary scores.
    "synchrony_score", "reciprocity", "circadian_flatness", "cross_account_dup_ratio",
]
META_FEATURES = [c for c in META_FEATURES if c in fusion.columns]
X_meta = fusion.loc[:, META_FEATURES].fillna(0.0).to_numpy(dtype=np.float64)
print(f"meta-learner features ({len(META_FEATURES)}): {META_FEATURES}")

_n_splits = int(min(5, np.bincount(y).min()))
if _n_splits < 2:
    raise RuntimeError("need at least 2 examples of each class to cross-validate the fusion")
cv = StratifiedKFold(n_splits=_n_splits, shuffle=True, random_state=settings.seed)

_meta = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=settings.seed),
)
strategies["stacking_logreg"] = cross_val_predict(
    _meta, X_meta, y, cv=cv, method="predict_proba"
)[:, 1]

# Single-branch references, so the fusion's lift is visible rather than asserted.
strategies["text_only"] = fusion["text_p90"].to_numpy()
strategies["graph_only"] = fusion["graph_score"].to_numpy()

print(f"\ncross-validated with {_n_splits}-fold stratified CV on {len(y)} accounts")

meta-learner features (14): ['graph_score', 'rf_score', 'text_p90', 'text_max', 'text_mean', 'text_top3_mean', 'text_flag_rate', 'text_std', 'has_text', 'n_posts', 'synchrony_score', 'reciprocity', 'circadian_flatness', 'cross_account_dup_ratio']

cross-validated with 5-fold stratified CV on 48 accounts


In [9]:
reports: dict[str, amx.ClassificationReport] = {}
thresholds: dict[str, float] = {}
for _name, _scores in strategies.items():
    _thr, _ = amx.tune_threshold(y, _scores, objective="fbeta", beta=FBETA)
    thresholds[_name] = _thr
    reports[_name] = amx.evaluate(y, _scores, threshold=_thr, beta=FBETA)

comparison = amx.compare_reports(reports)
comparison["threshold"] = [thresholds[n] for n in comparison.index]
print(comparison.to_string())

BEST = comparison["fbeta"].idxmax()
print(f"\nbest by F{FBETA}: {BEST}")
print(
    "\nIf `stacking_logreg` does not beat `weighted_avg` by a clear margin, prefer the"
    "\nweighted average: it has no fitted parameters, cannot overfit 48 accounts, and is"
    "\ntrivial to explain to an analyst. Complexity has to pay for itself."
)

11:46:56 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.2258 -> fbeta=0.7109 (0.5 would give 0.0000)


11:46:56 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.5017 -> fbeta=0.7109 (0.5 would give 0.3939)


11:46:56 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.3186 -> fbeta=1.0000 (0.5 would give 1.0000)


11:46:56 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.5017 -> fbeta=0.7109 (0.5 would give 0.3939)


11:46:56 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.0000 -> fbeta=0.5417 (0.5 would give 0.0000)


                 precision  recall        f1  delta_f1     fbeta   roc_auc    pr_auc       mcc     brier  accuracy  alert_rate     threshold  support  n_positive  tn  fp  fn  tp
model                                                                                                                                                                            
stacking_logreg   1.000000   1.000  1.000000  0.000000  1.000000  1.000000  1.000000  1.000000  0.006004  1.000000    0.166667  3.185511e-01       48           8  40   0   0   8
weighted_avg      0.500000   0.875  0.636364 -0.363636  0.710938  0.871875  0.615000  0.573944  0.142381  0.833333    0.291667  2.257720e-01       48           8  33   7   1   7
rule_or           0.500000   0.875  0.636364 -0.363636  0.710938  0.871875  0.615000  0.573944  0.251143  0.833333    0.291667  5.017155e-01       48           8  33   7   1   7
text_only         0.500000   0.875  0.636364 -0.363636  0.710938  0.871875  0.615000  0.573944  0.251143  0.83

### What the meta-learner learned

Coefficients from the model fitted on all rows. Sign and magnitude are interpretable
because the inputs are standardised. This is the table that answers *"what is the system
actually keying on?"*.

In [10]:
_meta.fit(X_meta, y)
_coefs = pd.Series(
    _meta.named_steps["logisticregression"].coef_[0], index=META_FEATURES
).sort_values(key=np.abs, ascending=False)
print(_coefs.round(3).to_string())

viz.plot_feature_importance(
    _coefs.index.tolist(), _coefs.abs().tolist(), top_n=len(_coefs),
    save_as=settings.paths.figures / "04_meta_coefficients.png",
)

synchrony_score            1.862
n_posts                    0.523
text_mean                  0.434
text_top3_mean             0.342
text_p90                   0.309
reciprocity                0.295
cross_account_dup_ratio   -0.286
circadian_flatness        -0.246
text_std                  -0.235
text_max                   0.217
text_flag_rate             0.178
rf_score                   0.112
graph_score               -0.059
has_text                   0.000


11:46:57 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\04_meta_coefficients.png


<Axes: title={'center': 'Top 14 features'}, xlabel='importance'>

## 5 · Calibration

The fused score is shown to an analyst as a *threat probability*, so it should behave like
one: among accounts scored 0.70, roughly 70% should be agents. A raw logistic output on
imbalanced, class-weighted data is usually over-confident.

Isotonic regression (`fusion_model.calibrate`) fixes the mapping without changing the
ranking, so AUC is unchanged while the numbers become meaningful. On a corpus this small
isotonic can overfit; the Brier score before and after is the check on that.

In [11]:
from sklearn.calibration import calibration_curve

raw_scores = strategies[BEST]
_calibrated_cv = cross_val_predict(
    CalibratedClassifierCV(_meta, method=str(FCFG.get("calibrate", "isotonic")), cv=3),
    X_meta, y, cv=cv, method="predict_proba",
)[:, 1] if BEST == "stacking_logreg" else raw_scores

_brier_raw = float(np.mean((raw_scores - y) ** 2))
_brier_cal = float(np.mean((_calibrated_cv - y) ** 2))
print(f"Brier  raw {_brier_raw:.4f}  ->  calibrated {_brier_cal:.4f}  "
      f"({'better' if _brier_cal < _brier_raw else 'WORSE — keep raw'})")

final_scores = _calibrated_cv if _brier_cal < _brier_raw else raw_scores
final_threshold, final_val = amx.tune_threshold(y, final_scores, objective="fbeta", beta=FBETA)
final_report = amx.evaluate(y, final_scores, threshold=final_threshold, beta=FBETA)

print(f"\nFINAL fused model ({BEST})")
print(f"  threshold {final_threshold:.3f}")
print(f"  {final_report}")
print(f"\n{amx.confusion_frame(final_report).to_string()}")

viz.plot_roc_pr(y, final_scores, save_as=settings.paths.figures / "04_fusion_roc_pr.png")
viz.plot_confusion(final_report, save_as=settings.paths.figures / "04_fusion_confusion.png")
viz.plot_score_distributions(y, final_scores, threshold=final_threshold,
                             save_as=settings.paths.figures / "04_fusion_scores.png")
# `show_bands` draws the four triage zones from §6 onto the sweep, so the operating
# point can be read against the bands an analyst actually acts on rather than as a
# bare number.
viz.plot_threshold_sweep(
    y, final_scores, beta=FBETA, chosen=final_threshold, show_bands=True, bands=BANDS,
    save_as=settings.paths.figures / "04_fusion_threshold_sweep.png",
)

Brier  raw 0.0060  ->  calibrated 0.0100  (WORSE — keep raw)
11:46:58 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.3186 -> fbeta=1.0000 (0.5 would give 1.0000)



FINAL fused model (stacking_logreg)
  threshold 0.319
  <report P=1.000 R=1.000 F1=1.000 F1.5=1.000 AUC=1.000 AP=1.000 @thr=0.32 n=48>

predicted            pred human/benign  pred adversarial  total
actual                                                         
actual human/benign                 40                 0     40
actual adversarial                   0                 8      8
total                               40                 8     48


11:46:58 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\04_fusion_roc_pr.png


11:46:58 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\04_fusion_confusion.png


11:46:59 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\04_fusion_scores.png


11:47:00 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\04_fusion_threshold_sweep.png


<Axes: title={'center': 'Operating point selection (F-beta, beta=1.5)'}, xlabel='decision threshold', ylabel='score'>

## 6 · Triage bands

A probability is not a decision. `fusion_model.decision_thresholds` turns the score into
the four actions an analyst can actually take:

| band | score | action |
|---|---|---|
| `clear` | < 0.35 | no action |
| `monitor` | 0.35 – 0.60 | watchlist, no human time |
| `investigate` | 0.60 – 0.82 | analyst review |
| `escalate` | ≥ 0.82 | immediate action |

The table below is the operational one: how many accounts land in each band, and what
fraction of each band is genuinely hostile (the band's precision). `escalate` needs high
precision or analysts stop trusting it; `monitor` can be noisy because nothing expensive
happens there.

In [12]:
fusion["fused_score"] = final_scores
fusion["band"] = amx.triage_bands(fusion["fused_score"], BANDS)

_band_table = (
    fusion.groupby("band", observed=False)
    .agg(accounts=("user_id", "size"),
         true_agents=("label", "sum"),
         precision=("label", "mean"),
         mean_score=("fused_score", "mean"))
    .round(3)
)
_band_table["pct_of_all_agents"] = (
    _band_table["true_agents"] / max(int(fusion["label"].sum()), 1)
).round(3)
print(_band_table.to_string())

_esc = fusion[fusion["band"] == "escalate"]
print(f"\nescalate band: {len(_esc)} accounts, "
      f"{100 * _esc['label'].mean() if len(_esc) else 0:.1f}% true agents")
print(f"agents NOT reaching 'investigate': "
      f"{int(((fusion['label'] == 1) & (~fusion['band'].isin(['investigate', 'escalate']))).sum())}")

             accounts  true_agents  precision  mean_score  pct_of_all_agents
band                                                                        
clear              40            0        0.0       0.040              0.000
monitor             0            0        NaN         NaN              0.000
investigate         1            1        1.0       0.764              0.125
escalate            7            7        1.0       0.949              0.875

escalate band: 7 accounts, 100.0% true agents
agents NOT reaching 'investigate': 0


In [13]:
print("highest-scoring accounts as the dashboard would list them:\n")
_display = (
    fusion.nlargest(12, "fused_score")
    .loc[:, ["user_id", "band", "fused_score", "text_p90", "graph_score",
             "synchrony_score", "n_posts", "label"]]
    .rename(columns={"label": "TRUE"})
    .round(3)
)
print(_display.to_string(index=False))

highest-scoring accounts as the dashboard would list them:

                 user_id        band  fused_score  text_p90  graph_score  synchrony_score  n_posts  TRUE
    agent::thriftymumof3    escalate        0.986     0.502          0.0            0.944       14     1
 agent::jord_wasdoubtful    escalate        0.985     0.502          0.0            0.944       18     1
  agent::orchestrator_00    escalate        0.981     0.502          0.0            0.691       23     1
   agent::liftwithcallum    escalate        0.972     0.502          0.0            0.925       17     1
agent::dr_reyna_holistic    escalate        0.962     0.502          0.0            0.716       13     1
    agent::mia_wellnesss    escalate        0.933     0.502          0.0            0.716       11     1
   agent::theglowjournal    escalate        0.824     0.502          0.0            0.738        8     1
agent::honest_reviews_uk investigate        0.764     0.502          0.0            0.777        3  

### Why each account fired

The explanation the dashboard's "why" panel renders. This is what late fusion buys and
joint training does not: the contribution of each branch is separable and can be stated
in a sentence.

In [14]:
def explain(row: pd.Series) -> str:
    """One-line, analyst-readable rationale for a fused score."""
    reasons = []
    if row["text_p90"] >= _tt:
        reasons.append(f"text {row['text_p90']:.2f} over {_tt:.2f} on {int(row['n_posts'])} posts")
    if row["graph_score"] >= _gt:
        reasons.append(f"network {row['graph_score']:.2f}")
    if row.get("synchrony_score", 0) > 0.5:
        reasons.append(f"synchrony {row['synchrony_score']:.2f} "
                       f"({int(row.get('synchrony_partner_count', 0))} partners)")
    if row.get("cross_account_dup_ratio", 0) > 0.3:
        reasons.append(f"content shared with neighbours {row['cross_account_dup_ratio']:.2f}")
    if row.get("circadian_flatness", 0) > 0.7:
        reasons.append("no diurnal rhythm")
    if not reasons:
        reasons.append("no single strong indicator — flagged on the combination")
    return "; ".join(reasons)


for _, _row in fusion.nlargest(6, "fused_score").iterrows():
    print(f"[{str(_row['band']).upper():<11}] {_row['user_id']}  score={_row['fused_score']:.3f}  "
          f"(true={'AGENT' if _row['label'] else 'organic'})")
    print(f"    {explain(_row)}\n")

[ESCALATE   ] agent::thriftymumof3  score=0.986  (true=AGENT)
    text 0.50 over 0.50 on 14 posts; synchrony 0.94 (5 partners); no diurnal rhythm

[ESCALATE   ] agent::jord_wasdoubtful  score=0.985  (true=AGENT)
    text 0.50 over 0.50 on 18 posts; synchrony 0.94 (6 partners); no diurnal rhythm

[ESCALATE   ] agent::orchestrator_00  score=0.981  (true=AGENT)
    text 0.50 over 0.50 on 23 posts; synchrony 0.69 (5 partners)

[ESCALATE   ] agent::liftwithcallum  score=0.972  (true=AGENT)
    text 0.50 over 0.50 on 17 posts; synchrony 0.92 (5 partners); no diurnal rhythm

[ESCALATE   ] agent::dr_reyna_holistic  score=0.962  (true=AGENT)
    text 0.50 over 0.50 on 13 posts; synchrony 0.72 (5 partners)

[ESCALATE   ] agent::mia_wellnesss  score=0.933  (true=AGENT)
    text 0.50 over 0.50 on 11 posts; synchrony 0.72 (2 partners)



## 7 · Does it hold on real accounts?

§4–6 are measured on generated data. This section repeats the fusion on **Cresci-2017**,
where the accounts and their tweets are real, by scoring each account's actual tweets with
the text branch and fusing with the graph score already computed in notebook 03.

The caveat to keep in view: the text model was trained on essays, QA answers and jailbreak
prompts, not on 2014 Italian tweets, so the text branch is far out of its domain here and
should be expected to contribute little. That in itself is the finding — the fused score
should not be *worse* than graph-only, which is the property a sound fusion layer must have.

In [15]:
_real_posts_path = settings.paths.processed / "graph_posts.parquet"
_real_text_path = settings.paths.processed / "graph_text_scores.parquet"

if _real_text_path.exists():
    real_text = iou.load_frame(_real_text_path)
    print(f"loaded cached scores for {len(real_text):,} real posts")
elif _real_posts_path.exists():
    from aegis.text_utils import injection_lexical_score

    _posts = iou.load_frame(_real_posts_path)
    # Cap per account: scoring 410k tweets with a transformer is a GPU job, and the
    # aggregators here (p90, top-3) are stable well below 200 posts per account.
    _posts = _posts.groupby("user_id", group_keys=False).head(40)
    print(f"scoring {len(_posts):,} real tweets (40/account cap)")
    try:
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        _tok = AutoTokenizer.from_pretrained(str(settings.paths.text_model))
        _mdl = AutoModelForSequenceClassification.from_pretrained(str(settings.paths.text_model))
        _mdl.eval()
        _out = []
        with torch.no_grad():
            for _i in range(0, len(_posts), 64):
                _b = _posts["text"].iloc[_i:_i + 64].astype(str).tolist()
                _enc = _tok(_b, truncation=True, max_length=256, padding=True, return_tensors="pt")
                _out.append(torch.softmax(_mdl(**_enc).logits, dim=-1)[:, 1].numpy())
        _posts["text_score"] = np.concatenate(_out)
    except Exception as exc:
        print(f"transformer unavailable ({type(exc).__name__}) — lexical proxy, lower bound.")
        _posts["text_score"] = _posts["text"].map(injection_lexical_score)
    real_text = _posts.loc[:, ["user_id", "text_score"]]
    iou.save_frame(real_text, _real_text_path)
else:
    real_text = pd.DataFrame(columns=["user_id", "text_score"])

loaded cached scores for 82,794 real posts


In [16]:
if len(real_text):
    real_agg = aggregate_text_to_account(real_text, score_col="text_score", user_col="user_id")
    real = graph_scores.merge(real_agg, on="user_id", how="left")
    real["has_text"] = real["text_mean"].notna().astype(int)
    for _c in _text_cols:
        real[_c] = real[_c].fillna(0.5 if _c != "text_std" else 0.0)
    real["n_posts"] = real["n_posts"].fillna(0)

    _test = real[real["split"] == "test"]
    _y_real = _test["label"].to_numpy()

    real_reports = {
        "text_only": amx.evaluate(_y_real, _test["text_p90"], beta=FBETA),
        "graph_only": amx.evaluate(_y_real, _test["graph_score"],
                                   threshold=_gt, beta=FBETA),
        "weighted_avg": amx.evaluate(
            _y_real,
            float(WEIGHTS["w_text"]) * _test["text_p90"].to_numpy()
            + float(WEIGHTS["w_graph"]) * _test["graph_score"].to_numpy(),
            beta=FBETA,
        ),
    }
    _Xr = real.loc[:, [c for c in META_FEATURES if c in real.columns]].fillna(0.0)
    _meta_real = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=settings.seed),
    )
    _meta_real.fit(_Xr[real["split"] == "train"], real.loc[real["split"] == "train", "label"])
    real_reports["stacking_logreg"] = amx.evaluate(
        _y_real, _meta_real.predict_proba(_Xr[real["split"] == "test"])[:, 1], beta=FBETA
    )

    print("Cresci-2017 test accounts (REAL data):")
    print(amx.compare_reports(real_reports).to_string())
    print(
        "\nThe text branch is far out of domain on 2014 Italian tweets, so it contributes"
        "\nlittle here. What matters is that fusion does not score BELOW graph-only — a"
        "\nfusion layer that can be dragged down by an uninformative branch is broken."
    )
else:
    real_reports = {}
    print("no real post scores available — skipping")

Cresci-2017 test accounts (REAL data):
                 precision    recall        f1  delta_f1     fbeta   roc_auc    pr_auc       mcc     brier  accuracy  alert_rate  threshold  support  n_positive   tn   fp  fn   tp
model                                                                                                                                                                              
stacking_logreg   1.000000  0.943396  0.970874  0.000000  0.960118  0.994923  0.996318  0.945980  0.021567  0.972308    0.461538        0.5      325         159  166    0   9  150
graph_only        0.974359  0.955975  0.965079 -0.005794  0.961557  0.995567  0.995957  0.932409  0.022776  0.966154    0.480000        0.5      325         159  162    4   7  152
weighted_avg      0.974359  0.955975  0.965079 -0.005794  0.961557  0.995567  0.995957  0.932409  0.067826  0.966154    0.480000        0.5      325         159  162    4   7  152
text_only         0.489231  1.000000  0.657025 -0.313849  0.7

## 8 · Robustness — what breaks this?

An attacker who knows the design will attack the weaker branch. Two adversarial
simulations, each degrading one branch and measuring what the other holds:

* **Text evasion** — the agent writes perfectly human prose. Simulated by pushing every
  agent's text score to the benign end.
* **Network evasion** — the agent operates alone with no coordination. Simulated by
  pushing every agent's graph score down.

The point of the fusion is that neither attack alone should collapse detection.

In [17]:
_scenarios = {}
_base = fusion.copy()

for _label, _col, _value in (
    ("text evasion (agents write like humans)", _text_cols, 0.05),
    ("network evasion (agents operate alone)", ["graph_score", "rf_score"], 0.05),
):
    _mod = _base.copy()
    _agent_rows = _mod["label"] == 1
    for _c in (_col if isinstance(_col, list) else [_col]):
        if _c in _mod.columns:
            _mod.loc[_agent_rows, _c] = _value
    _Xm = _mod.loc[:, META_FEATURES].fillna(0.0).to_numpy(dtype=np.float64)
    _s = _meta.predict_proba(_Xm)[:, 1]
    _scenarios[_label] = amx.evaluate(y, _s, threshold=final_threshold, beta=FBETA)

_scenarios["no evasion (baseline)"] = final_report
print(amx.compare_reports(_scenarios).to_string())
print(
    "\nRecall under each single-branch evasion is the headline robustness number. If it"
    "\nstays well above zero in both rows, an attacker must defeat text style AND"
    "\ncoordination structure at once — which is the whole point of the hybrid."
)

                                         precision  recall   f1  delta_f1  fbeta  roc_auc    pr_auc  mcc     brier  accuracy  alert_rate  threshold  support  n_positive  tn  fp  fn  tp
model                                                                                                                                                                                   
no evasion (baseline)                          1.0     1.0  1.0       0.0    1.0      1.0  1.000000  1.0  0.006004  1.000000    0.166667   0.318551       48           8  40   0   0   8
text evasion (agents write like humans)        0.0     0.0  0.0      -1.0    0.0      0.0  0.166667  0.0  0.168567  0.833333    0.000000   0.318551       48           8  40   0   8   0
network evasion (agents operate alone)         0.0     0.0  0.0      -1.0    0.0      0.0  0.166667  0.0  0.168567  0.833333    0.000000   0.318551       48           8  40   0   8   0

Recall under each single-branch evasion is the headline robustness number.

## 9 · Persist the fusion model

In [18]:
import joblib

_meta.fit(X_meta, y)
joblib.dump(_meta, settings.paths.fusion_model / "fusion_meta.joblib")
joblib.dump(
    {"features": META_FEATURES, "threshold": float(final_threshold),
     "bands": BANDS, "text_threshold": _tt, "graph_threshold": _gt,
     "weights": WEIGHTS, "strategy": BEST},
    settings.paths.fusion_model / "fusion_config.joblib",
)

iou.save_json(
    {
        "strategy": BEST,
        "meta_features": META_FEATURES,
        "threshold": float(final_threshold),
        "fbeta": FBETA,
        "bands": BANDS,
        "coefficients": {k: float(v) for k, v in _coefs.items()},
        "campaign_evaluation": {k: v.to_dict() for k, v in reports.items()},
        "campaign_final": final_report.to_dict(),
        "band_table": _band_table.reset_index().astype(str).to_dict("records"),
        "real_data_evaluation": {k: v.to_dict() for k, v in real_reports.items()},
        "robustness": {k: v.to_dict() for k, v in _scenarios.items()},
        "branch_correlation_spearman": float(_corr.loc["graph_score", "text_p90"]),
        "text_branch": text_meta.get("test"),
        "graph_branch": graph_meta.get("gnn_test"),
        "caveat": (
            "The fusion model is fitted and evaluated on the GENERATED 2026 campaign, "
            "because no public dataset provides both text and interaction-graph evidence "
            "over one population of real accounts. Section 7 is the real-data check. "
            "Report the campaign numbers as a simulation result, not a measurement."
        ),
    },
    settings.paths.fusion_model / "fusion_metrics.json",
)

iou.save_frame(
    fusion.loc[:, ["user_id", "label", "fused_score", "band", "text_p90", "graph_score",
                   "text_flag_rate", "n_posts", "has_text"]],
    settings.paths.processed / "fusion_scores.parquet",
)
print(f"artefacts -> {settings.paths.fusion_model}")

11:47:03 │ INFO    │ aegis.io │ wrote fusion_scores.parquet                  rows=48       cols=9   (7.3 KB)


artefacts -> C:\Users\dabhi\Documents\Major-Project\Complete-project\models\fusion_model


## 10 · End-to-end summary

The single table the project is judged on.

In [19]:
_final = pd.DataFrame([
    {"branch": "text (DeBERTa-v3)", "evaluated_on": "held-out text corpus",
     "f1": text_meta["test"]["f1"], "precision": text_meta["test"]["precision"],
     "recall": text_meta["test"]["recall"], "roc_auc": text_meta["test"]["roc_auc"]},
    *([{"branch": "graph (GraphSAGE)", "evaluated_on": f"{graph_meta['dataset']} test nodes",
        "f1": graph_meta["gnn_test"]["f1"], "precision": graph_meta["gnn_test"]["precision"],
        "recall": graph_meta["gnn_test"]["recall"], "roc_auc": graph_meta["gnn_test"]["roc_auc"]}]
      if graph_meta.get("gnn_test") else []),
    {"branch": f"HYBRID ({BEST})", "evaluated_on": "2026 synthetic campaign",
     "f1": final_report.f1, "precision": final_report.precision,
     "recall": final_report.recall, "roc_auc": final_report.roc_auc},
]).round(4)
print(_final.to_string(index=False))

print("\n" + "=" * 78)
print("CAVEATS THAT MUST TRAVEL WITH THESE NUMBERS")
print("=" * 78)
print(f"1. smoke_test={settings.smoke_test}"
      f"{' — corpora capped at %d rows; NOT publication numbers.' % settings.row_cap if settings.smoke_test else ''}")
print(f"2. The graph corpus is {graph_meta['label_assortativity']:.3f} label-assortative;")
print("   Cresci separates largely by community structure. Upper bound, not a result.")
print("3. The fusion layer is fitted on GENERATED campaign data (§7 is the real-data check).")
print("4. TweepFake, WildGuard, TwiBot-22 and TwiBot-24 were unavailable; see notebook 01 §2.")
print(f"5. Text branch cross-generator drop: "
      f"{(text_meta['test']['f1'] - text_meta['cross_generator_holdout']['report']['f1']):+.4f} F1"
      if text_meta.get("cross_generator_holdout", {}).get("report") else
      "5. Cross-generator holdout was skipped.")

                  branch            evaluated_on     f1  precision  recall  roc_auc
       text (DeBERTa-v3)    held-out text corpus 0.6882     0.5342  0.9669   0.6721
       graph (GraphSAGE)  cresci_2017 test nodes 0.9651     0.9744  0.9560   0.9956
HYBRID (stacking_logreg) 2026 synthetic campaign 1.0000     1.0000  1.0000   1.0000

CAVEATS THAT MUST TRAVEL WITH THESE NUMBERS
1. smoke_test=True — corpora capped at 1500 rows; NOT publication numbers.
2. The graph corpus is 0.958 label-assortative;
   Cresci separates largely by community structure. Upper bound, not a result.
3. The fusion layer is fitted on GENERATED campaign data (§7 is the real-data check).
4. TweepFake, WildGuard, TwiBot-22 and TwiBot-24 were unavailable; see notebook 01 §2.
5. Text branch cross-generator drop: +0.6882 F1


## Where this goes next

`models/` now contains everything the backend needs:

```
models/text_model/     fine-tuned DeBERTa-v3 + tokenizer + text_metrics.json
models/graph_model/    swarm_gnn.pt + feature_scaler.joblib + graph_metrics.json
models/fusion_model/   fusion_meta.joblib + fusion_config.joblib + fusion_metrics.json
```

**Phase 2** (`backend/`) loads these in `ml_service.py` and serves `/analyze_text`,
`/analyze_network` and `/get_threat_dashboard`. The `explain()` function in §6 becomes the
API's `reasons` field; the triage bands become its `band`.

**Phase 3** (`frontend/`) renders the band table as `MetricsPanel.jsx` and the campaign
graph from 03 §8 as `NetworkGraph.jsx`.